In [ ]:
!pip install -q --upgrade openai

In [ ]:
!pip install gradio

In [ ]:
import os
from IPython.display import Markdown, display
from openai import OpenAI
from google.colab import userdata
api = userdata.get('openaiapi')

In [ ]:
open_ai_client = OpenAI(api_key=api, base_url = "https://openrouter.ai/api/v1")

In [ ]:
def print_markdown(message):
  display(Markdown(message))

In [ ]:
def get_ai_tutor_response(user_question):
  system_prompt = "Your an AI Tutor that explain concepts clearly and concisely"
  try:
    response = open_ai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages = [
            {"role":"system", "content":system_prompt},
            {"role":"user", "content":user_question}
        ]
    )
    return response.choices[0].message.content
  except Exception as e:
    print(e)
    return f"Sorry, I encountered an error trying to get an answer: {e}"

In [ ]:
test_question = "Explain what a neural network is"
response = get_ai_tutor_response(test_question)
print_markdown(response)

A neural network is a computational model inspired by the way the human brain processes information. It consists of interconnected layers of nodes, or "neurons," which work together to analyze and understand data.

Here's a breakdown of the key components:

1. **Structure**: A neural network typically has three types of layers:
   - **Input Layer**: The first layer that receives raw data (features). Each node represents a feature of the input.
   - **Hidden Layers**: One or more layers between the input and output layers. These layers process the input data through weighted connections and activation functions, allowing the network to learn complex patterns.
   - **Output Layer**: The final layer that produces the output or prediction based on the processing done in the hidden layers.

2. **Weights and Biases**: Each connection between neurons has an associated weight, which adjusts as the model learns. Biases are added to help the network fit the data better.

3. **Activation Function**: After a neuron processes its input, it passes the result through an activation function, which determines whether the neuron should be activated (fired) or not. Common activation functions include ReLU (Rectified Linear Unit), Sigmoid, and Tanh.

4. **Training**: Neural networks learn from data through a process called training. During training, the network adjusts its weights and biases to minimize the difference between predicted outputs and actual outputs (the loss function) using an optimization algorithm like stochastic gradient descent.

5. **Applications**: Neural networks are widely used in various applications, including image recognition, natural language processing, game playing, and more.

In summary, a neural network is a powerful tool for modeling complex relationships in data, and it learns to make predictions or decisions by mimicking the way human brains operate.

In [ ]:
import gradio as gr

In [ ]:
expereince_level= {
    1: "like I'm 5 years old",
    2: "like I'm 10 years old",
    3: "like a high school student",
    4: "like a college student",
    5: "like an expert in the field",
}
def stream_ai_response_experience_level(user_question,expereincelevel):


  level = expereince_level.get(expereincelevel, "in a clear and concise way")
  system_prompt = f"You are a helpful AI Tutor. Explain the following concept {level}"
  try:
    stream = open_ai_client.chat.completions.create(
      model="gpt-4o-mini",
      messages = [
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_question}
      ],
      stream=True
  )
    text_chunk = ""
    for chunk in stream:
      if chunk.choices[0].delta and chunk.choices[0].delta.content:
        text_chunk += chunk.choices[0].delta.content
        yield text_chunk

  except Exception as e:
    print(e)
    yield f"Sorry, I encountered an error trying to get an answer: {e}"



In [ ]:
ai_tutor_interface = gr.Interface(fn=stream_ai_response_experience_level,
                                  inputs=[gr.Textbox(lines=2, placeholder="Ask me anything...", label="Your Question"),
                                          gr.Slider(minimum=1, maximum=5, step=1,value=3, label="Experience Level")],
                                  outputs=gr.Textbox(lines=2, label="AI Tutor Response", container = True),
                                  title="Your best AI Tutor",
                                  description= "Enter your question below and the AI Tutor will provide an explanation.",
                                  flagging_mode="never")
ai_tutor_interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://547fd21628a68995f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
